![Cabecera](../../../Sprint_11/assets/cabecera_thebridge.png)

# Workout — Multiagente con supervisor

## Objetivo

Montar un **overview completo y básico** del patrón **supervisor + especialistas** en LangGraph:

1. Varios nodos = varios “agentes” (prompts distintos).
2. Un **supervisor** decide el siguiente paso según el **estado compartido**.
3. Los especialistas **leen y escriben** en ese estado.
4. Hay **bucle** supervisor ↔ agentes y una salida clara (`FINISH` + tope de iteraciones).
5. **`MemorySaver` + `thread_id`**: el grafo guarda checkpoints por hilo (como en el bloque HITL).

```text
START → supervisor ──┬── cultura ────┐
                     ├── practico ───┼──► supervisor → … → FINISH → END
                     └── sintetizar ─┘
```

Dominio: planificar una **tarde cultural** (ideas + logística + cierre).

## Índice

1. Setup
2. Estado compartido
3. Supervisor, especialistas y router
4. Grafo con bucle + checkpointer (+ diagrama ASCII)
5. Ejecutar el pipeline (dos `thread_id`)
6. Para llevarnos

**Requisito:** `GEMINI_API_KEY`

## 1. Setup

Misma pila que los notebooks anteriores: LangGraph + Gemini vía LangChain.

Descomenta la celda de `%pip` solo la primera vez o si falta algún paquete.

In [ ]:
# %pip install -qU langgraph langchain-google-genai langchain-core python-dotenv

In [1]:
import os
from getpass import getpass

from dotenv import load_dotenv

load_dotenv()

if not os.getenv("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Pega tu GEMINI_API_KEY: ")

API_KEY = os.environ["GEMINI_API_KEY"]
print("GEMINI_API_KEY:", "sí" if API_KEY else "no")

GEMINI_API_KEY: sí


In [ ]:
from typing import Annotated, Literal, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import END, START, StateGraph

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=API_KEY,
    temperature=0,
)

MAX_ITER = 8  # tope duro en Python (el supervisor también lo mira)
print("LLM listo | MAX_ITER =", MAX_ITER)

LLM listo | MAX_ITER = 8


## 2. Estado compartido

El estado es el **canal entre agentes**: cada nodo lee lo que hay y escribe solo su parte.

| Campo | Quién lo rellena |
|-------|------------------|
| `pedido` | Entrada del usuario |
| `notas_cultura` / `notas_practico` | Especialistas |
| `respuesta` | Nodo `sintetizar` (plan final) |
| `siguiente` | Supervisor (`cultura` / `practico` / `sintetizar` / `FINISH`) |
| `iteracion` | Supervisor (contador) |
| `log` | Todos (reducer: se **acumula**) |

In [13]:
def _acumular_log(actual: list, nuevo: list) -> list:
    """Reducer de LangGraph: concatena listas en lugar de sobrescribir."""
    return (actual or []) + (nuevo or [])


class EstadoMulti(TypedDict):
    pedido: str
    notas_cultura: str
    notas_practico: str
    respuesta: str
    siguiente: str  # cultura | practico | sintetizar | FINISH
    iteracion: int
    log: Annotated[list, _acumular_log]


print("EstadoMulti definido")

EstadoMulti definido


## 3. Supervisor, especialistas y router

### Supervisor

No escribe el plan al usuario. Solo **mira el estado** y elige el siguiente nodo.
Si el LLM falla, un **fallback en Python** aplica reglas deterministas.
Si `iteracion >= MAX_ITER` → `FINISH` (evita bucles infinitos).

In [14]:
OPCIONES = ("cultura", "practico", "sintetizar", "FINISH")


def _fallback_siguiente(estado: EstadoMulti) -> str:
    """Reglas fijas si el LLM no devuelve una etiqueta válida."""
    if not (estado.get("notas_cultura") or "").strip():
        return "cultura"
    if not (estado.get("notas_practico") or "").strip():
        return "practico"
    if not (estado.get("respuesta") or "").strip():
        return "sintetizar"
    return "FINISH"


def nodo_supervisor(estado: EstadoMulti) -> dict:
    """Decide el siguiente agente (o FINISH) según el estado."""
    it = int(estado.get("iteracion") or 0)
    if it >= MAX_ITER:
        print(f"[supervisor] tope MAX_ITER={MAX_ITER} → FINISH")
        return {
            "siguiente": "FINISH",
            "iteracion": it + 1,
            "log": [f"iter {it + 1}: supervisor → FINISH (max_iter)"],
        }

    hay_c = "SÍ" if (estado.get("notas_cultura") or "").strip() else "NO"
    hay_p = "SÍ" if (estado.get("notas_practico") or "").strip() else "NO"
    hay_r = "SÍ" if (estado.get("respuesta") or "").strip() else "NO"

    prompt = [
        SystemMessage(
            content=(
                "Eres el supervisor de un equipo que planifica una tarde cultural.\n"
                "Agentes:\n"
                "- cultura: ideas de planes (museos, eventos, ambiente)\n"
                "- practico: horarios, transporte, presupuesto\n"
                "- sintetizar: une ambas notas en un plan final corto\n"
                "- FINISH: el plan final ya está listo\n\n"
                "Responde SOLO con una palabra: cultura, practico, sintetizar o FINISH.\n"
                "Reglas: sin notas_cultura → cultura; sin notas_practico → practico; "
                "con ambas y sin respuesta → sintetizar; con respuesta → FINISH."
            )
        ),
        HumanMessage(
            content=(
                f"Pedido: {estado.get('pedido', '')}\n"
                f"notas_cultura: {hay_c}\n"
                f"notas_practico: {hay_p}\n"
                f"respuesta: {hay_r}\n"
                f"iteracion: {it}"
            )
        ),
    ]
    raw = (llm.invoke(prompt).content or "").strip()
    # Normalizar etiqueta
    low = raw.lower()
    if "finish" in low:
        siguiente = "FINISH"
    elif "sintet" in low:
        siguiente = "sintetizar"
    elif "practic" in low:
        siguiente = "practico"
    elif "cultura" in low:
        siguiente = "cultura"
    else:
        siguiente = _fallback_siguiente(estado)

    if siguiente not in OPCIONES:
        siguiente = _fallback_siguiente(estado)

    print(f"[supervisor] iter={it + 1} → {siguiente}")
    return {
        "siguiente": siguiente,
        "iteracion": it + 1,
        "log": [f"iter {it + 1}: supervisor → {siguiente}"],
    }


print("nodo_supervisor listo")

nodo_supervisor listo


### Especialistas

- `cultura` / `practico`: escriben notas a partir del **pedido**.
- `sintetizar`: **lee** las dos notas y deja la `respuesta` final.

Así se ve que el estado compartido no es decoración: el sintetizador depende de lo que hicieron los otros.

In [5]:
def nodo_cultura(estado: EstadoMulti) -> dict:
    """Ideas culturales (museos, ambiente, eventos)."""
    print("[cultura] generando notas…")
    prompt = [
        SystemMessage(
            content=(
                "Eres especialista cultural. Da 3-5 bullets en español "
                "(ideas de planes, ambiente, tipo de visita). Sin horarios ni precios."
            )
        ),
        HumanMessage(content=estado["pedido"]),
    ]
    texto = llm.invoke(prompt).content or ""
    return {"notas_cultura": texto, "log": ["cultura: notas listas"]}


def nodo_practico(estado: EstadoMulti) -> dict:
    """Logística: transporte, presupuesto, timing."""
    print("[practico] generando notas…")
    prompt = [
        SystemMessage(
            content=(
                "Eres especialista práctico. Da 3-5 bullets en español "
                "(transporte, presupuesto orientativo, duración). Sin inventar precios oficiales."
            )
        ),
        HumanMessage(content=estado["pedido"]),
    ]
    texto = llm.invoke(prompt).content or ""
    return {"notas_practico": texto, "log": ["practico: notas listas"]}


def nodo_sintetizar(estado: EstadoMulti) -> dict:
    """Une notas_cultura + notas_practico en un plan final corto."""
    print("[sintetizar] uniendo notas…")
    prompt = [
        SystemMessage(
            content=(
                "Eres el cierre del equipo. Escribe un plan de tarde cultural en 5-8 líneas, "
                "usando SOLO las notas que te pasan. Español claro."
            )
        ),
        HumanMessage(
            content=(
                f"Pedido: {estado.get('pedido', '')}\n\n"
                f"Notas culturales:\n{estado.get('notas_cultura', '')}\n\n"
                f"Notas prácticas:\n{estado.get('notas_practico', '')}"
            )
        ),
    ]
    texto = llm.invoke(prompt).content or ""
    return {"respuesta": texto, "log": ["sintetizar: plan final listo"]}


print("Especialistas listos")

Especialistas listos


### Router

Tras cada visita al supervisor, LangGraph llama a esta función: lee `siguiente` y elige el nodo (o `END`).

In [6]:
def router_supervisor(
    estado: EstadoMulti,
) -> Literal["cultura", "practico", "sintetizar", "__end__"]:
    """Traduce la decisión del supervisor a un destino del grafo."""
    sig = estado.get("siguiente") or "cultura"
    if sig == "FINISH":
        return "__end__"
    if sig in ("cultura", "practico", "sintetizar"):
        return sig  # type: ignore[return-value]
    return "cultura"


print("router_supervisor listo")

router_supervisor listo


## 4. Grafo con bucle + checkpointer

Clave del overview:

- `START → supervisor`
- edges **condicionales** desde el supervisor
- cada especialista **vuelve** al supervisor
- `FINISH` → `END`
- **`MemorySaver`** al compilar: guarda un checkpoint del estado tras cada paso (en RAM)
- cada `invoke` llevará un **`thread_id`** (qué conversación / ejecución es)

Tras compilar, `draw_ascii()` muestra el bucle en texto.

In [7]:
from langgraph.checkpoint.memory import MemorySaver

grafo = StateGraph(EstadoMulti)
grafo.add_node("supervisor", nodo_supervisor)
grafo.add_node("cultura", nodo_cultura)
grafo.add_node("practico", nodo_practico)
grafo.add_node("sintetizar", nodo_sintetizar)

grafo.add_edge(START, "supervisor")
grafo.add_conditional_edges(
    "supervisor",
    router_supervisor,
    {
        "cultura": "cultura",
        "practico": "practico",
        "sintetizar": "sintetizar",
        "__end__": END,
    },
)
# Vuelta al supervisor = el bucle de orquestación
grafo.add_edge("cultura", "supervisor")
grafo.add_edge("practico", "supervisor")
grafo.add_edge("sintetizar", "supervisor")

# MemorySaver = checkpoints en RAM (se pierden al reiniciar el kernel)
app = grafo.compile(checkpointer=MemorySaver())
print("Grafo multiagente (bucle) + MemorySaver listo")
print()
# Diagrama ASCII: se ve el bucle especialista → supervisor
print(app.get_graph().draw_ascii())

Grafo multiagente (bucle) + MemorySaver listo

                                  +-----------+                                    
                                  | __start__ |                                    
                                  +-----------+                                    
                                        *                                          
                                        *                                          
                                        *                                          
                                  +------------+                                   
                                **| supervisor |..                                 
                          ******  +------------+  .....                            
                    ******       ...         ***       .......                     
              ******            .               *             .....                
          ****               

## 5. Ejecutar el pipeline

Con checkpointer, **cada** `invoke` necesita un `config` con `thread_id`.

Pasamos el estado inicial: `pedido` relleno; el resto vacío. El supervisor irá rellenando el trabajo a través de los especialistas.

Flujo esperado típico: `cultura` → `practico` → `sintetizar` → `FINISH`.

Después un **segundo pedido** con **otro** `thread_id` (misma app, conversación aislada).

In [8]:
estado_inicial: EstadoMulti = {
    "pedido": (
        "Quiero una tarde cultural en el centro: algo de museo o exposición, "
        "presupuesto contenido y moverme en transporte público."
    ),
    "notas_cultura": "",
    "notas_practico": "",
    "respuesta": "",
    "siguiente": "",
    "iteracion": 0,
    "log": [],
}

# thread_id = qué ejecución / conversación estamos guardando
config_1 = {"configurable": {"thread_id": "tarde-centro-001"}}
resultado = app.invoke(estado_inicial, config=config_1)

print("\n=== LOG ===")
for linea in resultado.get("log") or []:
    print(" •", linea)

print("\n=== PLAN FINAL ===")
print(resultado.get("respuesta") or "(sin respuesta)")
print("\niteraciones supervisor:", resultado.get("iteracion"))

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[supervisor] iter=1 → cultura
[cultura] generando notas…
[supervisor] iter=2 → practico
[practico] generando notas…
[supervisor] iter=3 → sintetizar
[sintetizar] uniendo notas…
[supervisor] iter=4 → FINISH

=== LOG ===
 • iter 1: supervisor → cultura
 • cultura: notas listas
 • iter 2: supervisor → practico
 • practico: notas listas
 • iter 3: supervisor → sintetizar
 • sintetizar: plan final listo
 • iter 4: supervisor → FINISH

=== PLAN FINAL ===
Aquí tienes tu plan de tarde cultural:

1.  Aprovecha las franjas horarias de entrada gratuita en museos emblemáticos (Reina Sofía, Prado), dedicando 2-3 horas a una exposición principal (coste 0€).
2.  Utiliza el transporte público (metro/autobús) para llegar, planificando tu ruta con antelación.
3.  Después, disfruta de un paseo a pie por los alrededores, admirando la arquitectura y el ambiente del centro.
4.  Para una pausa gastronómica auténtica y económica, busca opciones de comida callejera o mercados tradicionales.
5.  Permítete un de

### Checkpoint del hilo

`get_state(config)` lee el **último checkpoint** de ese `thread_id` sin volver a llamar al LLM.

In [9]:
snap = app.get_state(config_1)
estado_guardado = snap.values

print("thread_id:", config_1["configurable"]["thread_id"])
print("iteracion en checkpoint:", estado_guardado.get("iteracion"))
print("¿hay respuesta?:", bool((estado_guardado.get("respuesta") or "").strip()))
print()
print("--- notas_cultura (extracto) ---")
print((estado_guardado.get("notas_cultura") or "")[:400], "…\n")
print("--- notas_practico (extracto) ---")
print((estado_guardado.get("notas_practico") or "")[:400], "…")

thread_id: tarde-centro-001
iteracion en checkpoint: 4
¿hay respuesta?: True

--- notas_cultura (extracto) ---
¡Excelente elección! Una tarde cultural en el centro, aprovechando el transporte público y con presupuesto contenido, es una de las mejores maneras de conectar con la esencia de la ciudad. Aquí tienes algunas ideas:

*   **Inmersión Artística o Histórica con Horarios Estratégicos:** Aprovecha las franjas horarias de entrada gratuita en museos emblemáticos. Muchos museos nacionales ofrecen acceso l …

--- notas_practico (extracto) ---
¡Claro! Aquí tienes una guía práctica para tu tarde cultural en el centro, optimizando el presupuesto y el transporte público:

*   **Transporte:** El transporte público (metro, autobús) es tu mejor aliado para el centro. Te permite llegar directamente a la mayoría de los puntos de interés sin preocuparte por el aparcamiento. Planifica tu ruta con antelación usando apps como Google Maps o la app d …


### Segundo pedido (otro `thread_id`)

Mismo grafo + checkpointer, **otro** hilo. Si reutilizaras `tarde-centro-001`, el supervisor vería el plan ya cerrado y podría ir directo a `FINISH`.

Cambia el pedido (más “música / noche”) y comprueba que el `log` vuelve a pasar por los especialistas.

In [ ]:
estado_2: EstadoMulti = {
    "pedido": (
        "Prefiero música en vivo o un concierto pequeño por la tarde-noche, "
        "zona Lavapiés o alrededores, sin gastar mucho."
    ),
    "notas_cultura": "",
    "notas_practico": "",
    "respuesta": "",
    "siguiente": "",
    "iteracion": 0,
    "log": [],
}

# thread_id distinto = conversación vacía (no ve el checkpoint de tarde-centro-001)
config_2 = {"configurable": {"thread_id": "tarde-musica-002"}}
resultado_2 = app.invoke(estado_2, config=config_2)

print("=== LOG (pedido 2) ===")
for linea in resultado_2.get("log") or []:
    print(" •", linea)

print("\n=== PLAN FINAL (pedido 2) ===")
print(resultado_2.get("respuesta") or "(sin respuesta)")
print("\niteraciones supervisor:", resultado_2.get("iteracion"))

## 6. Para llevarnos

| Pieza | Rol |
|-------|-----|
| Estado compartido | Canal entre agentes (`notas_*`, `respuesta`, `log`) |
| Supervisor | Decide el **siguiente** paso; no hace el trabajo del especialista |
| Router | `add_conditional_edges` lee `siguiente` |
| Especialistas | Prompts distintos; escriben en el State |
| Bucle | Cada especialista **vuelve** al supervisor |
| `FINISH` + `MAX_ITER` | Parada explícita (LLM + guardrail en Python) |
| `MemorySaver` | Checkpoints en RAM tras cada paso |
| `thread_id` | Separa ejecuciones / conversaciones |
| `get_state(config)` | Lee el último checkpoint sin llamar al LLM |


En este workout has visto un multiagente **orquestado**: varios nodos con prompts distintos comparten un **estado** (notas → plan final).

El **supervisor** no escribe la respuesta: mira el estado, elige el siguiente rol y el grafo **vuelve** a él. Sin `FINISH` y `MAX_ITER`, el bucle podría no parar.

Con **`MemorySaver`**, cada `invoke` lleva un **`thread_id`** (mismo hilo = mismo checkpoint; otro hilo = ejecución aislada). `get_state` lee ese checkpoint sin llamar al LLM. Es el mismo mecanismo de checkpoint que en el bloque HITL; aquí no pausamos para aprobar, solo reutilizamos el hilo.

**Idea clave:** multiagente en LangGraph = **grafo + estado + orquestación + persistencia de hilo**, no varios chatbots sueltos hablando cada uno por su cuenta.


